# News Parser → ChatGPT → Google Sheets — запуск в Google Colab

Иди по ячейкам сверху вниз. Порядок такой:

1. **Шаг 0** — установка (клон репо + зависимости).
2. **Шаг A** — проверка парсинга без расходов (без OpenAI и без записи). Секреты НЕ нужны.
3. **Шаг B** — подключаем ChatGPT, смотрим нормализацию в логах (ещё без записи).
4. **Шаг C** — боевой запуск с записью в Google Sheets.

### Как хранить ключи (важно!)
НЕ вставляй ключи в ячейки. Слева нажми иконку 🔑 (Secrets) и добавь:
- `OPENAI_API_KEY` — твой **новый** ключ OpenAI;
- `GOOGLE_SHEET_ID` — ID таблицы;
- `GOOGLE_SERVICE_ACCOUNT_JSON` — весь JSON сервисного аккаунта одной строкой.

У каждого секрета включи тумблер «Notebook access».

> **Telegram** в Colab пока не подключаем — первая авторизация Telethon требует ввода кода. В списке источников сейчас только сайты, так что всё работает без Telegram.

## Шаг 0 — установка
Можно запускать повторно — ячейка сама перекачивает свежий код.

In [ ]:
%cd /content
!rm -rf tf-data-analysis-stat-task1-example
!git clone -b claude/awesome-archimedes-3yxyt8 https://github.com/maijesk/tf-data-analysis-stat-task1-example.git
%cd tf-data-analysis-stat-task1-example
!pip install -q -r requirements.txt
print('\nГотово: код и зависимости установлены.')

## Шаг A — проверка парсинга (бесплатно, без секретов)

`SKIP_OPENAI=true` — не вызываем ChatGPT. `DRY_RUN=true` — ничего не пишем в таблицу.

Смотри в логах, сколько новостей нашёл каждый источник и кто упал. Так мы поймём, какие из 53 сайтов живые.

In [ ]:
import os
os.environ['SKIP_OPENAI'] = 'true'    # не вызывать OpenAI
os.environ['DRY_RUN'] = 'true'        # не писать в Google Sheets
os.environ['MAX_ITEMS_PER_RUN'] = '0' # 0 = без ограничения
os.environ['LOG_LEVEL'] = 'INFO'
!python main.py

## Шаг B — подключаем ChatGPT (ещё без записи)

Нужен секрет `OPENAI_API_KEY`. Ограничиваем 10 публикациями, чтобы потратить копейки.
В логах увидишь строки вида `[A] AI для покупателя | авто | Заголовок`.

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_MODEL'] = 'gpt-4.1-mini'
os.environ['SKIP_OPENAI'] = 'false'
os.environ['DRY_RUN'] = 'true'         # всё ещё не пишем в таблицу
os.environ['MAX_ITEMS_PER_RUN'] = '10' # лимит ради экономии
!python main.py

## Шаг C — боевой запуск с записью в Google Sheets

Нужны секреты `OPENAI_API_KEY`, `GOOGLE_SHEET_ID`, `GOOGLE_SERVICE_ACCOUNT_JSON`.

⚠️ Не забудь **расшарить таблицу** на e-mail сервисного аккаунта (поле `client_email` в JSON) с правами **Editor**.
Листы `News` / `Raw` / `Skipped` создадутся автоматически.

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_MODEL'] = 'gpt-4.1-mini'
os.environ['GOOGLE_SHEET_ID'] = userdata.get('GOOGLE_SHEET_ID')
os.environ['GOOGLE_SERVICE_ACCOUNT_JSON'] = userdata.get('GOOGLE_SERVICE_ACCOUNT_JSON')
os.environ['SKIP_OPENAI'] = 'false'
os.environ['DRY_RUN'] = 'false'        # ТЕПЕРЬ пишем в таблицу
os.environ['MAX_ITEMS_PER_RUN'] = '50'
!python main.py